# A1 – Auditoría de calidad de datos
**Dueño:** Integrante 1 · **Segundo:** Integrante 5

Estado: **exploración** (perfil y detección). Todavía no se limpia nada: las decisiones están en `DECISIONES.md`, pendientes de validar con el profesor.

In [ ]:
import sys; sys.path.insert(0, "..")
import pandas as pd, numpy as np
from src.carga import cargar_crudo, RAW
d = cargar_crudo()
cli, trx, prd, dev, crm, cam = (d[k] for k in ["clientes","transacciones","productos","devoluciones","interacciones_crm","campanas_marketing"])

## 1. Perfil estructural
Se lee todo como texto (`dtype=str`) para que ningún valor extraño se convierta en NaN sin avisar. Se buscan vacíos, tokens de nulo disfrazados (`"null"`, `"N/A"`…), duplicados exactos y por ID, rangos numéricos y fechas no parseables.

In [ ]:
FECHAS = {"clientes":["fecha_registro"],"transacciones":["fecha"],"devoluciones":["fecha_devolucion"],
          "interacciones_crm":["fecha"],"campanas_marketing":["fecha_envio"]}
for n in d:
    t = pd.read_csv(RAW / f"{n}.csv", dtype=str, keep_default_na=False)
    idc = t.columns[0]
    print(f"\n{'='*70}\n{n}: {t.shape[0]} filas x {t.shape[1]} columnas")
    print(f"  duplicados exactos: {t.duplicated().sum()} | {idc} duplicados: {t[idc].duplicated().sum()}")
    for c in t.columns:
        s = t[c]; vac = (s.str.strip()=="").sum()
        tok = s.str.strip().str.lower().isin(["nan","null","none","n/a","na","-","?"]).sum()
        num = pd.to_numeric(s, errors="coerce")
        info = f"  {c:22s} vacíos={vac:5d} tokens_nulos={tok:4d} únicos={s.nunique():6d}"
        if num.notna().sum() > 0.9*(len(s)-vac) and num.notna().sum() > 0:
            info += f" | min={num.min():.3f} p50={num.median():.3f} max={num.max():.3f}"
        elif c in FECHAS.get(n, []):
            f = pd.to_datetime(s, errors="coerce", format="%Y-%m-%d")
            info += f" | {f.min().date()} → {f.max().date()} | no parseables={f.isna().sum()-vac}"
        elif s.nunique() <= 25:
            info += f" | {s.value_counts().to_dict()}"
        print(info)

## 2. Consistencia entre tablas
Chequeos que pide el enunciado (devoluciones sin transacción de origen, fechas imposibles) más los de integridad referencial y aritmética de precios.

In [ ]:
p = lambda t, v: print(f"{t:70s} {v}")
print("--- Claves huérfanas")
p("trx.cliente_id sin cliente", (~trx.cliente_id.isin(cli.cliente_id)).sum())
p("trx.producto_id sin producto", (~trx.producto_id.isin(prd.producto_id)).sum())
p("dev.transaccion_id sin transacción", (~dev.transaccion_id.isin(trx.transaccion_id)).sum())
p("dev/crm/cam cliente_id sin cliente", sum((~t.cliente_id.isin(cli.cliente_id)).sum() for t in (dev, crm, cam)))
p("clientes sin ninguna transacción", (~cli.cliente_id.isin(trx.cliente_id)).sum())

print("--- Transacciones")
calc = (trx.cantidad*trx.precio_unitario*(1-trx.descuento)).round(2)
p("monto_neto ≠ cant*precio*(1-desc) (tol. 0.05)", ((trx.monto_neto-calc).abs() > 0.05).sum())
m = trx.merge(prd[["producto_id","precio_lista","costo","categoria"]], on="producto_id")
p("precio_unitario ≠ precio_lista", ((m.precio_unitario-m.precio_lista).abs() > 0.01).sum())
p("venta neta por unidad < costo", (m.monto_neto/m.cantidad < m.costo).sum())
p("transacción antes del registro del cliente", (trx.merge(cli, on="cliente_id").eval("fecha < fecha_registro")).sum())

print("--- Productos")
mg = ((prd.precio_lista-prd.costo)/prd.precio_lista*100).round(1)
p("margen_bruto_pct incoherente con costo/precio", ((mg-prd.margen_bruto_pct).abs() > 0.15).sum())

print("--- Devoluciones")
dd = dev.merge(trx[["transaccion_id","cliente_id","fecha","monto_neto"]], on="transaccion_id", suffixes=("","_trx"))
p("devolución antes de la compra", (dd.fecha_devolucion < dd.fecha).sum())
p("monto_devuelto > monto_neto", (dd.monto_devuelto > dd.monto_neto+0.01).sum())
p("cliente de la devolución ≠ cliente de la compra", (dd.cliente_id != dd.cliente_id_trx).sum())
p("días compra→devolución (p50/p95/max)", (dd.fecha_devolucion-dd.fecha).dt.days.quantile([.5,.95,1]).tolist())
p("% transacciones con devolución", round(len(dev)/len(trx)*100, 2))

print("--- CRM")
q = crm.tiempo_respuesta_hrs; q1, q3 = q.quantile([.25,.75])
p("tiempo_respuesta > Q3+1.5·IQR", f"{(q > q3+1.5*(q3-q1)).sum()} (umbral {q3+1.5*(q3-q1):.1f} h)")
p("tiempo_respuesta > 168 h", (q > 168).sum())
p("interacción antes del registro", (crm.merge(cli, on="cliente_id").eval("fecha < fecha_registro")).sum())

print("--- Campañas")
p("convertido=1 con abierto=0", ((cam.convertido==1)&(cam.abierto==0)).sum())
p("campaña antes del registro", (cam.merge(cli, on="cliente_id").eval("fecha_envio < fecha_registro")).sum())
p("conversión global % / apertura %", (round(cam.convertido.mean()*100,2), round(cam.abierto.mean()*100,2)))

print("--- Segmento declarado vs comportamiento")
ult = trx.groupby("cliente_id").fecha.max()
c = cli.set_index("cliente_id").join(ult.rename("ultima"))
p("'Inactivo' con compra en últimos 90 días", ((c.segmento=="Inactivo")&(c.ultima >= trx.fecha.max()-pd.Timedelta(days=90))).sum())

## 3. Profundización de las 3 anomalías principales
1. ¿Las compras previas al registro se explican por el canal (compraba en tienda antes de registrarse online)?
2. ¿Qué hay detrás de las ventas bajo costo?
3. ¿Las conversiones de campaña tienen una compra real que las respalde?

In [ ]:
t = trx.merge(cli[["cliente_id","fecha_registro"]], on="cliente_id")
t["antes"] = t.fecha < t.fecha_registro
print("1) % de compras antes del registro por canal:\n", t.groupby("canal").antes.mean().round(3).to_string())
g = t.groupby("cliente_id").agg(r=("fecha_registro","first"), p=("fecha","min"))
print("   correlación fecha_registro vs primera compra:", round(g.r.map(pd.Timestamp.toordinal).corr(g.p.map(pd.Timestamp.toordinal)), 3))

b = m[m.monto_neto/m.cantidad < m.costo]
print("\n2) Ventas bajo costo — descuento min/p50/max:", b.descuento.quantile([0,.5,1]).round(3).tolist())
print("   por categoría:", b.categoria.value_counts().to_dict())

for ventana in (7, 14, 30):
    cv = cam[cam.convertido==1].merge(trx[["cliente_id","fecha"]], on="cliente_id", how="left")
    cv["d"] = (cv.fecha-cv.fecha_envio).dt.days
    ok = cv[cv.d.between(0, ventana)].campana_id.unique()
    tot = cam.loc[cam.convertido==1, "campana_id"]
    print(f"3) ventana {ventana:2d} días: {tot.isin(ok).sum()} de {len(tot)} conversiones con compra que las respalda")